# ⚠️ Project 10.3 — Fault Priority Triage
**DSA for Mechatronics · Week 10 — Sorting Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, time
from copy import deepcopy
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A SCADA system receives fault events, each with a **severity score** (1–100).
We need to:
1. Sort faults by severity using **quick sort** (with median-of-3 pivot, partition trace)
2. Find the **K-th most severe fault** without full sort using **QuickSelect** — O(n) average
3. Triage faults into three priority buckets (LOW / MEDIUM / HIGH) using
   **Dutch National Flag** — one-pass, O(n), in-place
4. Validate the **relative sort**: within each severity group, preserve original
   arrival order (stability analysis)


## Step 1 — Generate fault event log

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_FAULTS      = random.randint(16, 26)
K_SELECT      = random.randint(3, min(8, N_FAULTS // 2))
LOW_THRESH    = random.randint(25, 35)
HIGH_THRESH   = random.randint(65, 80)

FAULT_TYPES  = ["OVERHEAT","PRESSURE","VIBRATION","VOLTAGE","CURRENT",
                "LEAK","JAM","DISCONNECT","OVERLOAD","TIMEOUT"]

faults = []
for i in range(N_FAULTS):
    faults.append({
        "id"       : f"F{i+1:03d}",
        "type"     : random.choice(FAULT_TYPES),
        "severity" : random.randint(1, 100),
        "arrival"  : i,   # arrival order index
    })

print(f"Fault event log:")
print(f"  Total faults     : {N_FAULTS}")
print(f"  K (QuickSelect)  : {K_SELECT}  (find K-th most severe)")
print(f"  Priority bands   : LOW ≤ {LOW_THRESH}  |  {LOW_THRESH} < MEDIUM ≤ {HIGH_THRESH}  |  HIGH > {HIGH_THRESH}")
print()
print(f"  {'ID':<6}  {'Type':<12}  {'Severity':>9}  Priority")
print(f"  {'─'*6}  {'─'*12}  {'─'*9}  {'─'*8}")
for f in faults:
    prio = "HIGH" if f["severity"] > HIGH_THRESH else ("LOW" if f["severity"] <= LOW_THRESH else "MEDIUM")
    print(f"  {f['id']:<6}  {f['type']:<12}  {f['severity']:9d}  {prio}")


## Step 2 — Quick sort with partition trace

In [ ]:
def quick_sort_traced(arr_in, key=lambda x: x):
    """
    Quick sort with median-of-3 pivot.
    Records partition calls for trace output.
    """
    arr = arr_in[:]
    partition_log = []
    comps = [0]; swaps_qs = [0]

    def get(i): return key(arr[i])

    def median3(lo, hi):
        mid = (lo + hi) // 2
        if get(lo) > get(mid): arr[lo], arr[mid] = arr[mid], arr[lo]; swaps_qs[0]+=1
        if get(lo) > get(hi):  arr[lo], arr[hi]  = arr[hi],  arr[lo]; swaps_qs[0]+=1
        if get(mid) > get(hi): arr[mid], arr[hi] = arr[hi], arr[mid]; swaps_qs[0]+=1
        # put pivot at hi-1
        arr[mid], arr[hi] = arr[hi], arr[mid]; swaps_qs[0]+=1
        return key(arr[hi])

    def partition(lo, hi):
        if hi - lo < 2:
            comps[0] += 1
            if get(lo) > get(hi): arr[lo], arr[hi] = arr[hi], arr[lo]; swaps_qs[0]+=1
            return lo
        pivot = median3(lo, hi)
        i, j = lo, hi - 1
        while True:
            while i <= hi - 1:
                comps[0] += 1
                if get(i) >= pivot: break
                i += 1
            while j >= lo:
                comps[0] += 1
                if get(j) <= pivot: break
                j -= 1
            if i >= j: break
            arr[i], arr[j] = arr[j], arr[i]; swaps_qs[0]+=1; i+=1; j-=1
        arr[i], arr[hi] = arr[hi], arr[i]; swaps_qs[0]+=1
        partition_log.append((lo, hi, i, pivot))
        return i

    def sort(lo, hi):
        if lo < hi:
            p = partition(lo, hi)
            sort(lo, p - 1)
            sort(p + 1, hi)

    import sys; sys.setrecursionlimit(10000)
    sort(0, len(arr) - 1)
    return arr, comps[0], swaps_qs[0], partition_log

sev_list  = [f["severity"] for f in faults]
sorted_sev, qs_comps, qs_swaps, qs_log = quick_sort_traced(sev_list)

# Sort the fault objects by severity
sorted_faults = sorted(faults, key=lambda f: f["severity"])

print(f"Quick sort on {N_FAULTS} faults (severity values):")
print(f"  Comparisons : {qs_comps}")
print(f"  Swaps       : {qs_swaps}")
print(f"  Partitions  : {len(qs_log)}")
print()
print(f"  Partition trace (lo, hi → pivot_pos, pivot_val):")
for lo, hi, p, pv in qs_log[:10]:
    print(f"    [{lo:2d}, {hi:2d}] → pos {p:2d}, pivot={pv}")
if len(qs_log) > 10:
    print(f"    ... and {len(qs_log)-10} more partitions")
print()
print(f"  Sorted severities: {sorted_sev}")
print(f"  Verified vs sorted(): {'✅' if sorted_sev == sorted(sev_list) else '❌'}")


## Step 3 — QuickSelect: K-th most severe fault

In [ ]:
def quick_select(arr_in, k, key=lambda x: x):
    """
    QuickSelect: find the k-th LARGEST element in O(n) average time.
    (k=1 → largest, k=2 → second largest, …)

    Like quicksort but only recurse into the side containing position (n-k).
    target_rank = n - k  (0-indexed from smallest).

    Average O(n), worst O(n²) — random pivot avoids worst case.
    No full sort needed — saves significant work for large n.
    """
    arr = arr_in[:]
    target = len(arr) - k   # position of kth-largest in 0-indexed sorted order

    def partition(lo, hi):
        # random pivot for QuickSelect
        pivot_idx = random.randint(lo, hi)
        arr[pivot_idx], arr[hi] = arr[hi], arr[pivot_idx]
        pivot = key(arr[hi])
        i = lo
        for j in range(lo, hi):
            if key(arr[j]) <= pivot:
                arr[i], arr[j] = arr[j], arr[i]
                i += 1
        arr[i], arr[hi] = arr[hi], arr[i]
        return i

    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        p = partition(lo, hi)
        if p == target:
            return arr[p]
        elif p < target:
            lo = p + 1
        else:
            hi = p - 1
    return arr[lo]

kth_sev = quick_select(sev_list, K_SELECT)
kth_fault = next(f for f in sorted_faults[::-1] if f["severity"] == kth_sev)
true_kth  = sorted(sev_list, reverse=True)[K_SELECT - 1]

print(f"QuickSelect — finding the {K_SELECT}-th most severe fault:")
print(f"  Total faults     : {N_FAULTS}")
print(f"  k                : {K_SELECT}")
print(f"  QuickSelect result : severity = {kth_sev}")
print(f"  True k-th value    : severity = {true_kth}")
print(f"  Correct            : {'✅ YES' if kth_sev == true_kth else '❌ NO'}")
print()
print(f"  Top-{K_SELECT} most severe faults:")
for rank, f in enumerate(sorted_faults[::-1][:K_SELECT], 1):
    print(f"    {rank}. {f['id']} ({f['type']}) — severity {f['severity']}")


## Step 4 — Dutch National Flag triage (3-way partition)

In [ ]:
def dutch_national_flag(arr, low_t, high_t, key=lambda x: x):
    """
    Dutch National Flag — 3-way partition in ONE pass, O(n) time, O(1) extra space.

    Three pointers:
      lo  = everything below lo is LOW
      mid = current element being examined
      hi  = everything above hi is HIGH

    At each step examine arr[mid]:
      - key < low_t  → swap arr[lo] ↔ arr[mid], advance lo and mid
      - key > high_t → swap arr[mid] ↔ arr[hi], retreat hi  (don't advance mid!)
      - else         → just advance mid
    """
    a = arr[:]
    lo, mid, hi = 0, 0, len(a) - 1
    swaps_dnf = 0
    while mid <= hi:
        v = key(a[mid])
        if v <= low_t:
            a[lo], a[mid] = a[mid], a[lo]; swaps_dnf += 1
            lo += 1; mid += 1
        elif v > high_t:
            a[mid], a[hi] = a[hi], a[mid]; swaps_dnf += 1
            hi -= 1
        else:
            mid += 1
    return a, swaps_dnf

triaged, dnf_swaps = dutch_national_flag(
    faults, LOW_THRESH, HIGH_THRESH, key=lambda f: f["severity"])

low_g    = [f for f in triaged if f["severity"] <= LOW_THRESH]
med_g    = [f for f in triaged if LOW_THRESH < f["severity"] <= HIGH_THRESH]
high_g   = [f for f in triaged if f["severity"] > HIGH_THRESH]

print(f"Dutch National Flag triage:")
print(f"  Thresholds    : LOW ≤ {LOW_THRESH}  |  MEDIUM ({LOW_THRESH}–{HIGH_THRESH}]  |  HIGH > {HIGH_THRESH}")
print(f"  Swaps used    : {dnf_swaps}  (single pass!)")
print()
print(f"  LOW  ({len(low_g):2d} faults): ", end="")
print(", ".join(f"{f['id']}({f['severity']})" for f in low_g[:8]),
      "..." if len(low_g)>8 else "")
print(f"  MED  ({len(med_g):2d} faults): ", end="")
print(", ".join(f"{f['id']}({f['severity']})" for f in med_g[:8]),
      "..." if len(med_g)>8 else "")
print(f"  HIGH ({len(high_g):2d} faults): ", end="")
print(", ".join(f"{f['id']}({f['severity']})" for f in high_g[:8]),
      "..." if len(high_g)>8 else "")

# Verify partition correctness
all_low_ok  = all(f["severity"] <= LOW_THRESH    for f in low_g)
all_med_ok  = all(LOW_THRESH < f["severity"] <= HIGH_THRESH for f in med_g)
all_high_ok = all(f["severity"] > HIGH_THRESH    for f in high_g)
correct_dnf = all_low_ok and all_med_ok and all_high_ok
print(f"\n  Partition correct : {'✅ YES' if correct_dnf else '❌ NO'}")
print(f"  Total accounted   : {len(low_g)+len(med_g)+len(high_g)} / {N_FAULTS}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " FAULT PRIORITY TRIAGE — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  QUICK SORT" + " "*(W-12) + "║")
print(f"║  {'Faults sorted':<28}: {N_FAULTS:<26}║")
print(f"║  {'Comparisons':<28}: {qs_comps:<26}║")
print(f"║  {'Swaps':<28}: {qs_swaps:<26}║")
print(f"║  {'Partition steps':<28}: {len(qs_log):<26}║")
print("╠" + "═"*W + "╣")
print("║  QUICKSELECT" + " "*(W-13) + "║")
print(f"║  {'K (rank)':<28}: {K_SELECT:<26}║")
print(f"║  {'K-th severity found':<28}: {kth_sev:<26}║")
print(f"║  {'Correct':<28}: {'YES ✅' if kth_sev == true_kth else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  DUTCH NATIONAL FLAG" + " "*(W-21) + "║")
print(f"║  {'LOW faults (≤{LOW_THRESH})':<28}: {len(low_g):<26}║")
print(f"║  {'MEDIUM faults':<28}: {len(med_g):<26}║")
print(f"║  {'HIGH faults (>{HIGH_THRESH})':<28}: {len(high_g):<26}║")
print(f"║  {'Swaps (one pass)':<28}: {dnf_swaps:<26}║")
print(f"║  {'Partition correct':<28}: {'YES ✅' if correct_dnf else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about sorting in this context?

*Your answer here:*

---

**Q2 — Which sorting concept did you find most important, and why?**
Pick one algorithm or pattern (merge sort, quickselect, interval merging, counting sort…) and explain what problem it solves best.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
